In [11]:
from pipelinespec.utils.names import slug
from pathlib import Path
import pandas as pd
import json
import re

MODEL = Path('../data/results/gpt-6-luna')
SAMPLE = re.compile(r'_(\d+)$')
USAGE = ['input_tokens', 'output_tokens', 'total_tokens']

rows = []

for path in sorted(MODEL.glob('*/agents/**/*_agent*.json')):
    record = json.loads(path.read_text())
    sample = SAMPLE.search(path.stem)

    rows.append({
        'repo': path.relative_to(MODEL).parts[0],
        'sample': int(sample.group(1)) if sample else 1,
        **{key: record.get('usage', {}).get('total', {}).get(key, 0) for key in USAGE},
    })

manifest = json.loads(Path('../data/manifest.json').read_text())
size = pd.Series({slug(sample['name']): sample['jobs'] for sample in manifest}, name='jobs')

one_run = pd.DataFrame(rows).query('sample == 1')
per_repo = one_run.groupby('repo')[USAGE].sum()
per_repo['jobs'] = size.reindex(per_repo.index)

print(f'One run of the constraint agent on each of {len(per_repo)} repositories.')
display(per_repo[USAGE + ['jobs']].describe().round(0))
display(per_repo.sort_values('total_tokens'))

One run of the constraint agent on each of 30 repositories.


,input_tokens,output_tokens,total_tokens,jobs
count,30.0,30.0,30.0,30.0
mean,4034454.0,104293.0,4138747.0,31.0
std,5582437.0,114746.0,5686763.0,28.0
min,22611.0,824.0,23435.0,1.0
25%,536603.0,17956.0,554290.0,6.0
50%,1906065.0,69437.0,1968236.0,21.0
75%,5108176.0,168832.0,5287438.0,46.0
max,20957071.0,446699.0,21194455.0,95.0


,input_tokens,output_tokens,total_tokens,jobs
repo,,,,
gitignore,22611,824,23435,1
MoneyPrinterTurbo,183477,11101,194578,4
stable-diffusion-webui,207910,8348,216258,4
markitdown,225882,7232,233114,4
vue,269416,10532,279948,5
ponytail,335225,7870,343095,3
ohmyzsh,510394,15812,526206,6
awesome-llm-apps,520478,13959,534437,6
bootstrap,584979,28868,613847,14


In [12]:
INPUT_PER_MILLION = 0.10
OUTPUT_PER_MILLION = 0.50

per_repo['cost'] = per_repo['input_tokens'] / 1e6 * INPUT_PER_MILLION + per_repo['output_tokens'] / 1e6 * OUTPUT_PER_MILLION
per_repo['cost per job'] = per_repo['cost'] / per_repo['jobs']

display(per_repo[['cost', 'cost per job']].describe().round(3))
display(per_repo[['input_tokens', 'output_tokens', 'total_tokens', 'jobs', 'cost', 'cost per job']].sort_values('cost').round(3))

cost = per_repo['cost']
print(f'Running the constraint agent once on a repository costs ${cost.median():.2f} at the median (mean ${cost.mean():.2f}, range ${cost.min():.3f} to ${cost.max():.2f}).')

,cost,cost per job
count,30.000,30.000
mean,0.456,0.012
std,0.611,0.006
min,0.003,0.003
25%,0.063,0.008
50%,0.222,0.010
75%,0.600,0.013
max,2.296,0.030


,input_tokens,output_tokens,total_tokens,jobs,cost,cost per job
repo,,,,,,
gitignore,22611,824,23435,1,0.003,0.003
MoneyPrinterTurbo,183477,11101,194578,4,0.024,0.006
stable-diffusion-webui,207910,8348,216258,4,0.025,0.006
markitdown,225882,7232,233114,4,0.026,0.007
vue,269416,10532,279948,5,0.032,0.006
ponytail,335225,7870,343095,3,0.037,0.012
ohmyzsh,510394,15812,526206,6,0.059,0.010
awesome-llm-apps,520478,13959,534437,6,0.059,0.010
bootstrap,584979,28868,613847,14,0.073,0.005


Running the constraint agent once on a repository costs $0.22 at the median (mean $0.46, range $0.003 to $2.30).


In [13]:
diagnose = []

for saved in sorted(MODEL.glob('*/map.json')):
    repo = saved.parent.name

    for node in json.loads(saved.read_text())['jobs']:
        for test in node['tested']:
            for run in test['runs']:
                path = MODEL / repo / 'runs' / run / 'diagnose_agent.json'

                if path.exists():
                    usage = json.loads(path.read_text()).get('usage', {}).get('total', {})
                    diagnose.append({'repo': repo, **{key: usage.get(key, 0) for key in USAGE}})

diagnose = pd.DataFrame(diagnose)
diagnose['cost'] = diagnose['input_tokens'] / 1e6 * INPUT_PER_MILLION + diagnose['output_tokens'] / 1e6 * OUTPUT_PER_MILLION

per_diagnosis = diagnose[USAGE + ['cost']]
per_traced_repo = diagnose.groupby('repo').agg(diagnoses=('cost', 'size'), input_tokens=('input_tokens', 'sum'), output_tokens=('output_tokens', 'sum'), total_tokens=('total_tokens', 'sum'), cost=('cost', 'sum'))

display(per_diagnosis.describe().round(4))
display(per_traced_repo.sort_values('cost').round(3))

print(f'{len(diagnose)} diagnoses over {len(per_traced_repo)} traced repositories: ${diagnose["cost"].sum():.2f} in total, ${diagnose["cost"].mean():.4f} per diagnosis.')
print(f'Diagnosing a traced repository costs ${per_traced_repo["cost"].median():.3f} at the median (range ${per_traced_repo["cost"].min():.3f} to ${per_traced_repo["cost"].max():.3f}).')

,input_tokens,output_tokens,total_tokens,cost
count,79.0000,79.0000,79.0000,79.0000
mean,24894.2658,1198.4177,26092.6835,0.0031
std,23526.9198,1019.2599,24163.8162,0.0027
min,6280.0000,186.0000,6604.0000,0.0008
25%,10602.5000,418.0000,10810.5000,0.0013
50%,15974.0000,851.0000,16584.0000,0.0019
75%,26852.5000,1754.5000,29484.5000,0.0038
max,123190.0000,4476.0000,126608.0000,0.0140


,diagnoses,input_tokens,output_tokens,total_tokens,cost
repo,,,,,
MoneyPrinterTurbo,8,66362,3522,69884,0.008
markitdown,4,70040,2803,72843,0.008
vue,10,124026,5471,129497,0.015
ohmyzsh,11,123537,6674,130211,0.016
gitignore,5,121038,9484,130522,0.017
scrcpy,7,163817,3505,167322,0.018
iptv,8,279661,11055,290716,0.033
stable-diffusion-webui,17,356827,32314,389141,0.052
ponytail,9,661339,19847,681186,0.076


79 diagnoses over 9 traced repositories: $0.24 in total, $0.0031 per diagnosis.
Diagnosing a traced repository costs $0.017 at the median (range $0.008 to $0.076).
